# 5장. 분석을 믿을 수 있게 만드는 데이터 전처리

이 노트북은 `book/chapters/ch05_data_preprocessing.md` 강의안을 초보자가 그대로 따라 하며 이해할 수 있도록 구성한 실습 자료입니다.

이번 장의 핵심은 데이터를 무조건 깨끗하게 만드는 것이 아니라, 분석 결과를 신뢰할 수 있도록 데이터의 상태를 확인하고 처리 기준을 남기는 것입니다.


## 0. 이 노트북 사용 방법

아래 셀을 위에서부터 차례대로 실행하세요.

- 원본 데이터는 `data/raw/`에 그대로 둡니다.
- 전처리 결과는 `data/processed/`에 별도로 저장합니다.
- 전처리 요약 보고서는 `reports/ch05_preprocessing_summary.md`에 저장합니다.
- 코드가 실행되었다고 끝이 아니라, 처리 기준을 설명할 수 있어야 합니다.


## 1. 왜 전처리가 필요한가

현실의 데이터는 처음부터 분석하기 좋은 형태로 주어지지 않습니다.

예를 들어 다음과 같은 문제가 있을 수 있습니다.

| 문제 유형 | 예시 | 확인할 질문 |
|---|---|---|
| 결측치 | 나이가 비어 있는 고객 | 비어 있는 이유가 무엇인가? 삭제해도 되는가? |
| 중복 | 같은 고객 ID가 여러 번 등장 | 자연스러운 중복인가, 데이터 오류인가? |
| 타입 오류 | 가격이 문자열로 저장됨 | 계산 가능한 숫자형으로 바꿀 수 있는가? |
| 날짜 오류 | 주문일이 문자열로 저장됨 | 월별·요일별 분석에 사용할 수 있는가? |
| 문자열 표기 차이 | `Seoul`, ` Seoul`, `SEOUL` | 같은 값을 하나의 표기로 통일해야 하는가? |
| 이상값 | 수량이 음수이거나 가격이 0원 | 실제 의미가 있는 값인가, 입력 오류인가? |
| 파생 컬럼 필요 | 수량과 단가만 있고 주문금액이 없음 | 분석에 필요한 새 컬럼을 만들 수 있는가? |


## 2. 원본 데이터와 전처리 데이터 분리하기

전처리에서 가장 중요한 원칙은 원본 데이터를 직접 덮어쓰지 않는 것입니다.

```text
data/
├─ raw/
│  ├─ customers.csv
│  ├─ products.csv
│  ├─ orders.csv
│  └─ order_items.csv
└─ processed/
   ├─ customers_clean.csv
   ├─ products_clean.csv
   ├─ orders_clean.csv
   └─ order_items_clean.csv
```

원본과 전처리 결과를 분리하면 실수했을 때 원본으로 돌아갈 수 있고, 전처리 전후 차이를 비교할 수 있습니다.


## 3. 패키지와 경로 설정

노트북이 `notebooks/` 폴더 안에서 실행되는 경우와 프로젝트 루트에서 실행되는 경우를 모두 고려해 경로를 설정합니다.


In [1]:
from pathlib import Path
import sys

import pandas as pd
import numpy as np

CURRENT_DIR = Path.cwd()

def find_project_root(start):
    for candidate in (start, *start.parents):
        if (candidate / 'pyproject.toml').is_file() and (candidate / 'chapter05').is_dir():
            return candidate
    raise FileNotFoundError('프로젝트 루트를 찾을 수 없습니다.')

PROJECT_ROOT = find_project_root(CURRENT_DIR)
RAW_DIR = PROJECT_ROOT / 'practice' / 'chapter05' / 'data' / 'raw'
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
REPORT_DIR = PROJECT_ROOT / 'reports'

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print('현재 실행 위치:', CURRENT_DIR)
print('프로젝트 루트:', PROJECT_ROOT)
print('Chapter 05 원본 데이터 폴더:', RAW_DIR)
print('전처리 데이터 폴더:', PROCESSED_DIR)
print('보고서 폴더:', REPORT_DIR)

현재 실행 위치: /home/gem/research/4. 학교 수업/LLM 수업/llm-data-analysis-study
프로젝트 루트: /home/gem/research/4. 학교 수업/LLM 수업/llm-data-analysis-study
Chapter 05 원본 데이터 폴더: /home/gem/research/4. 학교 수업/LLM 수업/llm-data-analysis-study/practice/chapter05/data/raw
전처리 데이터 폴더: /home/gem/research/4. 학교 수업/LLM 수업/llm-data-analysis-study/data/processed
보고서 폴더: /home/gem/research/4. 학교 수업/LLM 수업/llm-data-analysis-study/reports


## 4. 원본 데이터 불러오기

이번 장에서는 온라인 쇼핑몰 예제 데이터 4개를 사용합니다.

- `customers.csv`: 고객 정보
- `products.csv`: 상품 정보
- `orders.csv`: 주문 정보
- `order_items.csv`: 주문 상세 정보


In [2]:
customers = pd.read_csv(RAW_DIR / 'customers.csv')
products = pd.read_csv(RAW_DIR / 'products.csv')
orders = pd.read_csv(RAW_DIR / 'orders.csv')
order_items = pd.read_csv(RAW_DIR / 'order_items.csv')

print('원본 데이터 불러오기 완료')


원본 데이터 불러오기 완료


## 5. 전처리 전 데이터 크기 기록하기

전처리 전 데이터 크기를 기록해 두면 나중에 어떤 데이터가 얼마나 바뀌었는지 비교할 수 있습니다.


In [3]:
raw_data = {
    'customers': customers,
    'products': products,
    'orders': orders,
    'order_items': order_items,
}

raw_shapes = pd.DataFrame({
    'dataset': list(raw_data.keys()),
    'rows': [df.shape[0] for df in raw_data.values()],
    'columns': [df.shape[1] for df in raw_data.values()],
})

raw_shapes


,dataset,rows,columns
0,customers,152,6
1,products,102,4
2,orders,302,5
3,order_items,765,5


In [4]:
for name, df in raw_data.items():
    print(f'[{name}]')
    print('shape:', df.shape)
    print('columns:', list(df.columns))
    print()


[customers]
shape: (152, 6)
columns: ['customer_id', 'name', 'gender', 'age', 'city', 'signup_date']

[products]
shape: (102, 4)
columns: ['product_id', 'product_name', 'category', 'price']

[orders]
shape: (302, 5)
columns: ['order_id', 'customer_id', 'order_date', 'payment_method', 'order_status']

[order_items]
shape: (765, 5)
columns: ['order_item_id', 'order_id', 'product_id', 'quantity', 'unit_price']



## 6. 원본을 복사해서 전처리 시작하기

전처리 과정에서는 원본 DataFrame을 직접 수정하지 않고 복사본을 사용합니다.


In [5]:
customers_clean = customers.copy()
products_clean = products.copy()
orders_clean = orders.copy()
order_items_clean = order_items.copy()

clean_data = {
    'customers': customers_clean,
    'products': products_clean,
    'orders': orders_clean,
    'order_items': order_items_clean,
}

print('전처리용 복사본 생성 완료')


전처리용 복사본 생성 완료


## 7. 결측치 확인하기

결측치를 처리하기 전에 먼저 어디에 얼마나 비어 있는 값이 있는지 확인합니다. 개수와 비율을 함께 보면 데이터 크기에 따른 차이를 이해하기 쉽습니다.


In [6]:
for name, df in clean_data.items():
    print(f'\n[{name}] 결측치 개수')
    print(df.isna().sum())



[customers] 결측치 개수
customer_id    0
name           0
gender         0
age            4
city           2
signup_date    0
dtype: int64

[products] 결측치 개수
product_id      0
product_name    0
category        0
price           0
dtype: int64

[orders] 결측치 개수
order_id          0
customer_id       0
order_date        0
payment_method    0
order_status      0
dtype: int64

[order_items] 결측치 개수
order_item_id    0
order_id         0
product_id       0
quantity         0
unit_price       0
dtype: int64


In [7]:
def missing_summary(df):
    summary = pd.DataFrame({
        'missing_count': df.isna().sum(),
        'missing_ratio': (df.isna().mean() * 100).round(2),
    })
    return summary.sort_values('missing_count', ascending=False)

missing_summary(customers_clean)


,missing_count,missing_ratio
age,4,2.63
city,2,1.32
name,0,0.00
customer_id,0,0.00
gender,0,0.00
signup_date,0,0.00


## 8. 결측치 처리하기

결측 처리 전에는 **원래부터 비어 있던 값**과 숫자 변환 때문에 **새롭게 결측이 된 값**을 분리해 확인합니다. 나이는 중앙값으로 대체하고, 도시는 `Unknown`으로 구분합니다.


In [8]:
if 'age' in customers_clean.columns:
    age_before_conversion = customers_clean['age'].copy()
    age_numeric = pd.to_numeric(age_before_conversion, errors='coerce')
    age_original_missing = age_before_conversion.isna()
    age_new_conversion_failure = age_before_conversion.notna() & age_numeric.isna()
    age_median = age_numeric.median()

    age_conversion_audit = pd.DataFrame([
        {'check': '원래 age 결측', 'count': int(age_original_missing.sum())},
        {'check': '숫자 변환으로 새로 생긴 결측', 'count': int(age_new_conversion_failure.sum())},
        {'check': '대체 전 전체 age 결측', 'count': int(age_numeric.isna().sum())},
    ])
    display(age_conversion_audit)
    print('신규 변환 실패 원문값:')
    display(age_before_conversion.loc[age_new_conversion_failure].value_counts())
    print('age 중앙값:', age_median)

    customers_clean['age'] = age_numeric.fillna(age_median)

if 'city' in customers_clean.columns:
    customers_clean['city'] = customers_clean['city'].fillna('Unknown')

display(missing_summary(customers_clean))


,check,count
0,원래 age 결측,4
1,숫자 변환으로 새로 생긴 결측,2
2,대체 전 전체 age 결측,6


신규 변환 실패 원문값:


age
unknown    2
Name: count, dtype: int64

age 중앙값: 41.0


,missing_count,missing_ratio
customer_id,0,0.0
name,0,0.0
gender,0,0.0
age,0,0.0
city,0,0.0
signup_date,0,0.0


## 9. 중복 확인하기

중복은 맥락에 따라 의미가 다릅니다. `customers`의 `customer_id` 중복은 오류일 가능성이 크지만, `order_items`의 `order_id` 반복은 한 주문에 여러 상품이 들어갈 수 있으므로 자연스럽습니다.


In [9]:
for name, df in clean_data.items():
    print(name, '전체 행 중복 수:', df.duplicated().sum())


customers 전체 행 중복 수: 2
products 전체 행 중복 수: 2
orders 전체 행 중복 수: 2
order_items 전체 행 중복 수: 2


In [10]:
key_checks = {
    'customers': ('customer_id', customers_clean),
    'products': ('product_id', products_clean),
    'orders': ('order_id', orders_clean),
    'order_items': ('order_item_id', order_items_clean),
}

for name, (key_col, df) in key_checks.items():
    if key_col in df.columns:
        print(name, key_col, '중복 수:', df[key_col].duplicated().sum())
    else:
        print(name, key_col, '컬럼 없음')


customers customer_id 중복 수: 2
products product_id 중복 수: 2
orders order_id 중복 수: 2
order_items order_item_id 중복 수: 2


In [11]:
customers_clean = customers_clean.drop_duplicates()
products_clean = products_clean.drop_duplicates()
orders_clean = orders_clean.drop_duplicates()
order_items_clean = order_items_clean.drop_duplicates()

print('완전 중복 행 제거 완료')


완전 중복 행 제거 완료


## 10. 문자열 표기 정리하기

문자열 데이터에는 앞뒤 공백이 숨어 있을 수 있습니다. `Seoul`과 ` Seoul `은 눈으로는 비슷하지만 pandas에서는 서로 다른 값입니다.

결측치를 문자열 `nan`으로 바꾸지 않도록 주의하면서 문자열 컬럼의 앞뒤 공백을 제거합니다.


In [12]:
def strip_string_columns(df):
    result = df.copy()
    string_columns = result.select_dtypes(include=['object', 'string']).columns

    for col in string_columns:
        result[col] = result[col].where(
            result[col].isna(),
            result[col].astype(str).str.strip()
        )

    return result

customers_clean = strip_string_columns(customers_clean)
products_clean = strip_string_columns(products_clean)
orders_clean = strip_string_columns(orders_clean)
order_items_clean = strip_string_columns(order_items_clean)

print('문자열 앞뒤 공백 제거 완료')


문자열 앞뒤 공백 제거 완료


In [13]:
if 'city' in customers_clean.columns:
    display(customers_clean['city'].value_counts().head(10))

if 'order_status' in orders_clean.columns:
    display(orders_clean['order_status'].value_counts())


city
부산    23
대구    21
광주    21
서울    18
울산    17
인천    16
수원    10
성남     9
고양     8
대전     5
Name: count, dtype: int64

order_status
completed    175
cancelled     58
refunded      52
Complete       2
cancel         2
Refund         2
complete       1
COMPLETED      1
완료             1
Cancel         1
CANCELLED      1
취소             1
refund         1
REFUNDED       1
환불             1
Name: count, dtype: int64

## 11. 주문 상태값 통일하기

주문 상태값이 `complete`, `Complete`, `COMPLETED`, `완료`처럼 섞여 있으면 같은 상태를 서로 다른 값으로 집계할 수 있습니다. 대표 표기로 통일합니다.


In [14]:
status_map = {
    'complete': 'completed',
    'Complete': 'completed',
    'COMPLETED': 'completed',
    '완료': 'completed',
    'cancel': 'cancelled',
    'Cancel': 'cancelled',
    'CANCELLED': 'cancelled',
    '취소': 'cancelled',
    'refund': 'refunded',
    'Refund': 'refunded',
    'REFUNDED': 'refunded',
    '환불': 'refunded',
}

if 'order_status' in orders_clean.columns:
    orders_clean['order_status'] = orders_clean['order_status'].replace(status_map)
    display(orders_clean['order_status'].value_counts())


order_status
completed    180
cancelled     63
refunded      57
Name: count, dtype: int64

## 12. 날짜 컬럼 변환하기

날짜처럼 보이는 값도 실제로는 문자열일 수 있습니다. 월별 주문 분석이나 요일별 분석을 하려면 날짜형으로 변환해야 합니다.

`errors="coerce"`는 변환할 수 없는 값을 `NaT`로 바꿉니다. 그래서 변환 후 실패 건수를 반드시 확인해야 합니다.


In [15]:
orders_clean['order_date'] = pd.to_datetime(orders_clean['order_date'], errors='coerce')
print('order_date 변환 실패:', orders_clean['order_date'].isna().sum())
print('주문 시작일:', orders_clean['order_date'].min())
print('주문 종료일:', orders_clean['order_date'].max())

orders_clean['order_month'] = orders_clean['order_date'].dt.to_period('M').astype(str)
orders_clean['order_dayofweek'] = orders_clean['order_date'].dt.day_name()

orders_clean[['order_date', 'order_month', 'order_dayofweek']].head()


order_date 변환 실패: 3
주문 시작일: 2025-07-01 00:00:00
주문 종료일: 2028-12-31 00:00:00


,order_date,order_month,order_dayofweek
0,2026-07-12,2026-07,Sunday
1,2025-10-02,2025-10,Thursday
2,2026-03-13,2026-03,Friday
3,2025-10-12,2025-10,Sunday
4,2026-04-17,2026-04,Friday


In [16]:
if 'signup_date' in customers_clean.columns:
    customers_clean['signup_date'] = pd.to_datetime(customers_clean['signup_date'], errors='coerce')
    print('signup_date 변환 실패:', customers_clean['signup_date'].isna().sum())
    display(customers_clean[['customer_id', 'signup_date']].head())


signup_date 변환 실패: 2


,customer_id,signup_date
0,1,2025-10-05
1,2,2023-09-20
2,3,2025-01-22
3,4,2025-03-13
4,5,2024-02-10


## 13. 숫자형 컬럼 변환하기

가격, 수량, 단가는 숫자처럼 보여도 문자열일 수 있습니다. 특히 `10,000`처럼 쉼표가 들어간 값은 바로 계산하기 어렵습니다.


In [17]:
def to_number(series):
    return pd.to_numeric(
        series.astype(str).str.replace(',', '', regex=False),
        errors='coerce'
    )

products_clean['price'] = to_number(products_clean['price'])
order_items_clean['quantity'] = to_number(order_items_clean['quantity'])
order_items_clean['unit_price'] = to_number(order_items_clean['unit_price'])

print('price 변환 실패:', products_clean['price'].isna().sum())
print('quantity 변환 실패:', order_items_clean['quantity'].isna().sum())
print('unit_price 변환 실패:', order_items_clean['unit_price'].isna().sum())


price 변환 실패: 2
quantity 변환 실패: 5
unit_price 변환 실패: 2


## 14. 이상값 후보 확인하기

이상값은 일반적인 범위를 벗어난 값입니다. 하지만 이상값이 항상 오류는 아닙니다. 0원 상품은 이벤트 상품일 수 있고, 음수 수량은 반품을 의미할 수도 있습니다. 먼저 확인하고 기준을 정해야 합니다.


In [18]:
display(products_clean['price'].describe())
display(order_items_clean[['quantity', 'unit_price']].describe())


count    9.800000e+01
mean     2.005102e+05
std      1.001829e+06
min     -2.000000e+04
25%      5.200000e+04
50%      9.250000e+04
75%      1.527500e+05
max      9.999999e+06
Name: price, dtype: float64

,quantity,unit_price
count,758.000000,761.000000
mean,3.168865,103494.086728
std,4.159055,56969.823769
min,-1.000000,0.000000
25%,2.000000,55000.000000
50%,3.000000,96000.000000
75%,4.000000,154000.000000
max,100.000000,197000.000000


In [19]:
print('price <= 0:', len(products_clean[products_clean['price'] <= 0]))
print('quantity <= 0:', len(order_items_clean[order_items_clean['quantity'] <= 0]))
print('unit_price <= 0:', len(order_items_clean[order_items_clean['unit_price'] <= 0]))


price <= 0: 2
quantity <= 0: 3
unit_price <= 0: 1


### 실습용 이상값 처리 기준

이번 보완에서는 엔터티와 거래 행을 구분합니다.

```text
- 상품 가격의 변환 실패·비양수 값: 상품 행은 유지하고 price만 결측 + 품질 상태 기록
- quantity·unit_price 변환 실패·비양수 주문상세: 분석용 clean에서 제외하고 quarantine 보존
```

상품 행을 가격 오류만으로 삭제하면 정상 `product_id`까지 사라져 새로운 FK orphan이 생길 수 있기 때문입니다.


In [20]:
# 상품은 엔터티 행을 유지하고 가격만 검토 대상으로 표시합니다.
price_conversion_failed = products_clean['price'].isna()
price_nonpositive = products_clean['price'].notna() & products_clean['price'].le(0)
products_clean['price_quality_status'] = 'valid'
products_clean.loc[price_conversion_failed, 'price_quality_status'] = 'conversion_failed'
products_clean.loc[price_nonpositive, 'price_quality_status'] = 'nonpositive_review'
products_clean.loc[price_conversion_failed | price_nonpositive, 'price'] = np.nan

# 거래 계산에 사용할 수 없는 주문상세는 삭제하지 않고 별도 보류합니다.
invalid_item_value = (
    order_items_clean['quantity'].isna()
    | order_items_clean['unit_price'].isna()
    | order_items_clean['quantity'].le(0)
    | order_items_clean['unit_price'].le(0)
)
order_items_value_quarantine = order_items_clean.loc[invalid_item_value].copy()
order_items_value_quarantine['quarantine_reason'] = 'invalid_quantity_or_unit_price'
order_items_clean = order_items_clean.loc[~invalid_item_value].copy()

print('가격 품질 상태:')
display(products_clean['price_quality_status'].value_counts().rename('count'))
print('숫자값 문제로 보류한 order_items:', len(order_items_value_quarantine))
print('처리 후 products:', products_clean.shape)
print('처리 후 order_items:', order_items_clean.shape)


가격 품질 상태:


price_quality_status
valid                 96
conversion_failed      2
nonpositive_review     2
Name: count, dtype: int64

숫자값 문제로 보류한 order_items: 11
처리 후 products: (100, 5)
처리 후 order_items: (752, 5)


## 15. 파생 컬럼 만들기

전처리된 수량과 단가를 사용하면 주문 상세 금액을 계산할 수 있습니다.

`line_total = quantity × unit_price`


In [21]:
order_items_clean['line_total'] = (
    order_items_clean['quantity'] * order_items_clean['unit_price']
)

order_items_clean[['quantity', 'unit_price', 'line_total']].head()


,quantity,unit_price,line_total
0,2.0,172000.0,344000.0
1,4.0,79000.0,316000.0
2,3.0,87000.0,261000.0
3,1.0,187000.0,187000.0
4,5.0,125000.0,625000.0


In [22]:
print('전처리 후 주문 상세 금액 합계:', order_items_clean['line_total'].sum())


전처리 후 주문 상세 금액 합계: 252140000.0


## 16. 파일 간 관계 다시 확인하고 불일치 행 보류하기

먼저 전처리 중간 단계의 FK 불일치를 확인합니다. 그다음 원래부터 관계가 맞지 않는 주문·주문상세를 별도 quarantine으로 옮기고, 최종 clean 4개 테이블의 FK가 모두 0건인지 다시 검증합니다.


In [23]:
def relationship_table(customers_df, products_df, orders_df, order_items_df):
    return pd.DataFrame({
        'check': [
            'orders.customer_id exists in customers.customer_id',
            'order_items.order_id exists in orders.order_id',
            'order_items.product_id exists in products.product_id',
        ],
        'invalid_count': [
            int((~orders_df['customer_id'].isin(customers_df['customer_id'])).sum()),
            int((~order_items_df['order_id'].isin(orders_df['order_id'])).sum()),
            int((~order_items_df['product_id'].isin(products_df['product_id'])).sum()),
        ],
    })

pre_relationship_checks = relationship_table(
    customers_clean, products_clean, orders_clean, order_items_clean
)
print('보류 전 관계 검증')
display(pre_relationship_checks)

invalid_customer_mask = ~orders_clean['customer_id'].isin(customers_clean['customer_id'])
orders_quarantine = orders_clean.loc[invalid_customer_mask].copy()
orders_quarantine['quarantine_reason'] = 'customer_id_not_found'
orders_clean = orders_clean.loc[~invalid_customer_mask].copy()

invalid_order_mask = ~order_items_clean['order_id'].isin(orders_clean['order_id'])
invalid_product_mask = ~order_items_clean['product_id'].isin(products_clean['product_id'])
relationship_quarantine_mask = invalid_order_mask | invalid_product_mask
order_items_relationship_quarantine = order_items_clean.loc[relationship_quarantine_mask].copy()
order_items_relationship_quarantine['quarantine_reason'] = [
    ';'.join(reason for condition, reason in [
        (order_bad, 'order_id_not_found'),
        (product_bad, 'product_id_not_found'),
    ] if condition)
    for order_bad, product_bad in zip(
        invalid_order_mask.loc[relationship_quarantine_mask],
        invalid_product_mask.loc[relationship_quarantine_mask],
    )
]
order_items_clean = order_items_clean.loc[~relationship_quarantine_mask].copy()

order_items_all_quarantine = pd.concat(
    [order_items_value_quarantine, order_items_relationship_quarantine],
    ignore_index=True,
    sort=False,
).drop_duplicates(subset=['order_item_id'], keep='first')

quarantine = {
    'orders': orders_quarantine,
    'order_items': order_items_all_quarantine,
}
quarantine_summary = pd.DataFrame([
    {'reason': 'orders.customer_id_not_found', 'quarantined_rows': int(invalid_customer_mask.sum())},
    {'reason': 'order_items.invalid_quantity_or_unit_price', 'quarantined_rows': len(order_items_value_quarantine)},
    {'reason': 'order_items.order_id_not_found', 'quarantined_rows': int(invalid_order_mask.sum())},
    {'reason': 'order_items.product_id_not_found', 'quarantined_rows': int(invalid_product_mask.sum())},
    {'reason': 'unique_order_items_quarantined_total', 'quarantined_rows': len(order_items_all_quarantine)},
])

relationship_checks = relationship_table(
    customers_clean, products_clean, orders_clean, order_items_clean
)
print('보류 요약')
display(quarantine_summary)
print('최종 clean 관계 검증')
display(relationship_checks)
assert relationship_checks['invalid_count'].eq(0).all()
print('FINAL FK CONTRACT: PASS')


보류 전 관계 검증


,check,invalid_count
0,orders.customer_id exists in customers.custome...,2
1,order_items.order_id exists in orders.order_id,2
2,order_items.product_id exists in products.prod...,2


보류 요약

,reason,quarantined_rows
0,orders.customer_id_not_found,2
1,order_items.invalid_quantity_or_unit_price,11
2,order_items.order_id_not_found,6
3,order_items.product_id_not_found,2
4,unique_order_items_quarantined_total,18


최종 clean 관계 검증


,check,invalid_count
0,orders.customer_id exists in customers.custome...,0
1,order_items.order_id exists in orders.order_id,0
2,order_items.product_id exists in products.prod...,0


FINAL FK CONTRACT: PASS


## 17. 전처리 전후 비교하기

전처리 전후 데이터 크기를 비교합니다. 행 수가 줄었다면 어떤 기준으로 줄었는지 설명할 수 있어야 하고, 열 수가 늘었다면 어떤 파생 컬럼이 추가되었는지 설명할 수 있어야 합니다.


In [24]:
processed_data = {
    'customers': customers_clean,
    'products': products_clean,
    'orders': orders_clean,
    'order_items': order_items_clean,
}

processed_shapes = pd.DataFrame({
    'dataset': list(processed_data.keys()),
    'rows': [df.shape[0] for df in processed_data.values()],
    'columns': [df.shape[1] for df in processed_data.values()],
})

comparison = raw_shapes.merge(
    processed_shapes,
    on='dataset',
    suffixes=('_raw', '_processed')
)

comparison


,dataset,rows_raw,columns_raw,rows_processed,columns_processed
0,customers,152,6,150,6
1,products,102,4,100,5
2,orders,302,5,298,7
3,order_items,765,5,745,6


## 18. 전처리 결과 저장하기

전처리된 데이터는 `data/processed` 폴더에 저장합니다. Excel에서 한글 CSV를 바로 열 수 있도록 `utf-8-sig` 인코딩을 사용합니다.


In [25]:
from src.preprocessing import (
    save_quarantine_data,
    validate_saved_processed_data,
)

customers_clean.to_csv(PROCESSED_DIR / 'customers_clean.csv', index=False, encoding='utf-8-sig')
products_clean.to_csv(PROCESSED_DIR / 'products_clean.csv', index=False, encoding='utf-8-sig')
orders_clean.to_csv(PROCESSED_DIR / 'orders_clean.csv', index=False, encoding='utf-8-sig')
order_items_clean.to_csv(PROCESSED_DIR / 'order_items_clean.csv', index=False, encoding='utf-8-sig')

quarantine_paths = save_quarantine_data(
    quarantine,
    PROCESSED_DIR / 'quarantine',
)

reload_validation = validate_saved_processed_data(
    expected_data=processed_data,
    output_dir=PROCESSED_DIR,
)
display(reload_validation)
assert reload_validation['status'].eq('PASS').all()
print('SAVED CSV RELOAD VALIDATION: PASS')


,check,value,status
0,customers.shape,"(150, 6)",PASS
1,customers.columns,6,PASS
2,customers.customer_id_unique_nonnull,0,PASS
3,products.shape,"(100, 5)",PASS
4,products.columns,5,PASS
5,products.product_id_unique_nonnull,0,PASS
6,orders.shape,"(298, 7)",PASS
7,orders.columns,7,PASS
8,orders.order_id_unique_nonnull,0,PASS
9,order_items.shape,"(745, 6)",PASS


SAVED CSV RELOAD VALIDATION: PASS


## 19. 전처리 요약 보고서 저장하기

전처리 기준과 결과를 Markdown 파일로 남겨 두면 이후 보고서 작성이 쉬워집니다.


In [26]:
summary_text = f'''# Chapter 5 데이터 전처리 요약

## 전처리 결과 파일

- customers_clean.csv
- products_clean.csv
- orders_clean.csv
- order_items_clean.csv

## 전처리 전후 데이터 크기

```text
{comparison.to_string(index=False)}
```

## 관계 불일치 보류 결과

```text
{quarantine_summary.to_string(index=False)}
```

## 최종 파일 간 관계 점검

```text
{relationship_checks.to_string(index=False)}
```

## 저장 후 CSV 재로딩 검증

```text
{reload_validation.to_string(index=False)}
```

## 주요 판단

- age 원래 결측과 신규 숫자 변환 실패를 분리하여 확인
- 가격 오류 상품은 삭제하지 않고 가격 품질 상태로 보존
- 관계 불일치 행은 quarantine에 보존하고 clean FK는 0건으로 확정
- 저장된 CSV를 다시 읽어 shape, columns, PK, 날짜, 계산식, FK 검증
'''

report_path = REPORT_DIR / 'ch05_preprocessing_summary.md'
report_path.write_text(summary_text, encoding='utf-8')
print('요약 보고서 저장 완료:', report_path)


요약 보고서 저장 완료: /home/gem/research/4. 학교 수업/LLM 수업/llm-data-analysis-study/reports/ch05_preprocessing_summary.md


## 20. 전처리 과정을 함수로 정리하기

전처리 코드를 한 번만 실행하고 끝내면 재사용하기 어렵습니다. 같은 데이터를 다시 받거나 처리 기준을 조금 바꾸어야 할 때는 함수로 정리된 코드가 훨씬 유용합니다.


In [27]:
def preprocess_customers(df):
    result = strip_string_columns(df)
    if 'age' in result.columns:
        numeric_age = pd.to_numeric(result['age'], errors='coerce')
        result['age'] = numeric_age.fillna(numeric_age.median())
    if 'city' in result.columns:
        result['city'] = result['city'].fillna('Unknown')
    if 'signup_date' in result.columns:
        result['signup_date'] = pd.to_datetime(result['signup_date'], errors='coerce')
    return result.drop_duplicates()


def preprocess_products(df):
    result = strip_string_columns(df)
    if 'price' in result.columns:
        numeric_price = to_number(result['price'])
        conversion_failed = result['price'].notna() & numeric_price.isna()
        nonpositive = numeric_price.notna() & numeric_price.le(0)
        result['price'] = numeric_price
        result['price_quality_status'] = 'valid'
        result.loc[conversion_failed, 'price_quality_status'] = 'conversion_failed'
        result.loc[nonpositive, 'price_quality_status'] = 'nonpositive_review'
        result.loc[conversion_failed | nonpositive, 'price'] = np.nan
    return result.drop_duplicates()


def preprocess_orders(df):
    result = strip_string_columns(df)
    if 'order_status' in result.columns:
        result['order_status'] = result['order_status'].replace(status_map)
    if 'order_date' in result.columns:
        result['order_date'] = pd.to_datetime(result['order_date'], errors='coerce')
        result['order_month'] = result['order_date'].dt.to_period('M').astype(str)
        result['order_dayofweek'] = result['order_date'].dt.day_name()
    return result.drop_duplicates()


def preprocess_order_items(df):
    result = strip_string_columns(df)
    result['quantity'] = to_number(result['quantity'])
    result['unit_price'] = to_number(result['unit_price'])
    valid = result['quantity'].gt(0) & result['unit_price'].gt(0)
    result = result.loc[valid].copy()
    result['line_total'] = result['quantity'] * result['unit_price']
    return result.drop_duplicates()


def enforce_relationships(data):
    result = {name: frame.copy() for name, frame in data.items()}
    result['orders'] = result['orders'].loc[
        result['orders']['customer_id'].isin(result['customers']['customer_id'])
    ].copy()
    result['order_items'] = result['order_items'].loc[
        result['order_items']['order_id'].isin(result['orders']['order_id'])
        & result['order_items']['product_id'].isin(result['products']['product_id'])
    ].copy()
    return result


In [28]:
manual_stage_data = {
    'customers': preprocess_customers(customers),
    'products': preprocess_products(products),
    'orders': preprocess_orders(orders),
    'order_items': preprocess_order_items(order_items),
}
manual_processed_data = enforce_relationships(manual_stage_data)

for name, frame in manual_processed_data.items():
    print(name, frame.shape)

display(relationship_table(
    manual_processed_data['customers'],
    manual_processed_data['products'],
    manual_processed_data['orders'],
    manual_processed_data['order_items'],
))


customers

 (150, 6)
products (100, 5)
orders (298, 7)
order_items (745, 6)


,check,invalid_count
0,orders.customer_id exists in customers.custome...,0
1,order_items.order_id exists in orders.order_id,0
2,order_items.product_id exists in products.prod...,0


## 21. 소스 모듈 사용하기

위에서 직접 작성한 전처리 함수는 `src/preprocessing.py`에도 정리되어 있습니다. 반복 작업이나 실제 프로젝트에서는 노트북 안에만 코드를 두기보다 소스 모듈로 분리하는 것이 좋습니다.


In [29]:
from src.data_loader import load_sales_data
from src.preprocessing import (
    build_preprocessing_report,
    compare_shapes,
    duplicate_summary,
    preprocess_sales_data_with_audit,
    save_processed_data,
    validate_relationships,
    validate_saved_processed_data,
)

module_raw_data = load_sales_data(RAW_DIR)
(
    module_processed_data,
    module_quarantine,
    module_quarantine_summary,
) = preprocess_sales_data_with_audit(module_raw_data)
module_relationship_checks = validate_relationships(module_processed_data)
module_comparison = compare_shapes(module_raw_data, module_processed_data)
module_reload_validation = validate_saved_processed_data(
    module_processed_data, PROCESSED_DIR
)

display(module_comparison)


,dataset,rows_raw,columns_raw,rows_processed,columns_processed
0,customers,152,6,150,6
1,order_items,765,5,745,6
2,orders,302,5,298,7
3,products,102,4,100,5


In [30]:
duplicate_summary(
    module_processed_data,
    key_columns={
        'customers': 'customer_id',
        'products': 'product_id',
        'orders': 'order_id',
        'order_items': 'order_item_id',
    },
)


,dataset,row_duplicate_count,key_column,key_duplicate_count
0,customers,0,customer_id,0
1,products,0,product_id,0
2,orders,0,order_id,0
3,order_items,0,order_item_id,0


In [31]:
display(module_relationship_checks)
display(module_quarantine_summary)
display(module_reload_validation)
assert module_relationship_checks['invalid_count'].eq(0).all()
assert module_reload_validation['status'].eq('PASS').all()


,check,invalid_count
0,orders.customer_id exists in customers.custome...,0
1,order_items.order_id exists in orders.order_id,0
2,order_items.product_id exists in products.prod...,0


,reason,quarantined_rows
0,orders.customer_id_not_found,2
1,order_items.invalid_quantity_or_unit_price,11
2,order_items.order_id_not_found,6
3,order_items.product_id_not_found,2
4,unique_order_items_quarantined_total,18


,check,value,status
0,customers.shape,"(150, 6)",PASS
1,customers.columns,6,PASS
2,customers.customer_id_unique_nonnull,0,PASS
3,products.shape,"(100, 5)",PASS
4,products.columns,5,PASS
5,products.product_id_unique_nonnull,0,PASS
6,orders.shape,"(298, 7)",PASS
7,orders.columns,7,PASS
8,orders.order_id_unique_nonnull,0,PASS
9,order_items.shape,"(745, 6)",PASS


## 22. 스크립트로 한 번에 실행하기

노트북에서 한 단계씩 이해한 전처리 과정을 스크립트로도 실행할 수 있습니다. 터미널에서 프로젝트 루트 기준으로 아래 명령을 실행합니다.

```bash
python scripts/preprocess_data.py
```

이 스크립트는 전처리 결과 CSV와 요약 보고서를 자동으로 저장합니다.


In [32]:
import hashlib
import subprocess

tracked_outputs = [
    PROCESSED_DIR / 'customers_clean.csv',
    PROCESSED_DIR / 'products_clean.csv',
    PROCESSED_DIR / 'orders_clean.csv',
    PROCESSED_DIR / 'order_items_clean.csv',
]

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

hashes_before = {path.name: sha256(path) for path in tracked_outputs}
script_result = subprocess.run(
    [sys.executable, str(PROJECT_ROOT / 'scripts' / 'preprocess_data.py')],
    cwd=PROJECT_ROOT,
    capture_output=True,
    text=True,
    check=True,
)
print(script_result.stdout)
hashes_after = {path.name: sha256(path) for path in tracked_outputs}

script_reload_validation = pd.read_csv(REPORT_DIR / 'ch05_reload_validation.csv')
script_quarantine_summary = pd.read_csv(REPORT_DIR / 'ch05_quarantine_summary.csv')
reproducibility_check = pd.DataFrame([
    {
        'file': name,
        'same_hash_after_script': hashes_before[name] == hashes_after[name],
    }
    for name in hashes_before
])
display(reproducibility_check)
display(script_reload_validation)
display(script_quarantine_summary)
assert reproducibility_check['same_hash_after_script'].all()
assert script_reload_validation['status'].eq('PASS').all()
print('INDEPENDENT SCRIPT + RELOAD VALIDATION: PASS')


전처리 완료
입력 Raw 경로: /home/gem/research/4. 학교 수업/LLM 수업/llm-data-analysis-study/practice/chapter05/data/raw

[전처리 전후 데이터 크기]
    dataset  rows_raw  columns_raw  rows_processed  columns_processed
  customers       152            6             150                  6
order_items       765            5             745                  6
     orders       302            5             298                  7
   products       102            4             100                  5

[중복 점검 결과]
    dataset  row_duplicate_count    key_column  key_duplicate_count
  customers                    0   customer_id                    0
   products                    0    product_id                    0
     orders                    0      order_id                    0
order_items                    0 order_item_id                    0

[파일 간 관계 점검 결과]
                                               check  invalid_count
  orders.customer_id exists in customers.customer_id              0
      order_items.order

,file,same_hash_after_script
0,customers_clean.csv,True
1,products_clean.csv,True
2,orders_clean.csv,True
3,order_items_clean.csv,True


,check,value,status
0,customers.shape,"(150, 6)",PASS
1,customers.columns,6,PASS
2,customers.customer_id_unique_nonnull,0,PASS
3,products.shape,"(100, 5)",PASS
4,products.columns,5,PASS
5,products.product_id_unique_nonnull,0,PASS
6,orders.shape,"(298, 7)",PASS
7,orders.columns,7,PASS
8,orders.order_id_unique_nonnull,0,PASS
9,order_items.shape,"(745, 6)",PASS


,reason,quarantined_rows
0,orders.customer_id_not_found,2
1,order_items.invalid_quantity_or_unit_price,11
2,order_items.order_id_not_found,6
3,order_items.product_id_not_found,2
4,unique_order_items_quarantined_total,18


INDEPENDENT SCRIPT + RELOAD VALIDATION: PASS


## 23. LLM과 함께 전처리 코드를 검토하기

LLM에게 전처리 코드를 요청할 때는 실제 고객명, 이메일, 주문 내역을 그대로 입력하지 않는 것이 좋습니다. 컬럼명, 데이터 타입, 결측치 개수, 중복 여부, 처리 목적처럼 구조화된 정보만 제공하세요.

```text
다음 전처리 코드가 안전한지 검토해 주세요.

customers["age"] = customers["age"].fillna(customers["age"].mean())
customers = customers.drop_duplicates()
orders["order_date"] = pd.to_datetime(orders["order_date"])
order_items["line_total"] = order_items["quantity"] * order_items["unit_price"]

검토 기준:
- 원본 데이터를 직접 수정하는 문제가 있는지
- 결측치 처리 방식이 적절한지
- 날짜 변환 실패를 확인하는지
- 중복 제거 기준이 충분한지
- line_total 계산 전에 숫자형 변환이 필요한지
- 더 안전한 코드로 어떻게 수정할 수 있는지
```


### LLM 코드 검토 기록

- **제안:** 평균 대체, 전체 중복 제거, 날짜의 즉시 변환, 숫자형 확인 전 `line_total` 계산.
- **검증:** 원본 보존, 변환 실패 건수, ID별 중복 의미, 숫자 변환 선행 여부를 실제 데이터로 확인했다.
- **판단:** 원본 복사본 사용, 나이는 중앙값 대체, 날짜는 `errors="coerce"` 후 실패 건수 기록, 중복은 주요 키와 함께 점검하는 방식으로 수정·채택했다.

## 24. 실습 과제

아래 과제를 직접 해결해 보세요.

1. 각 데이터셋의 결측치 비율을 하나의 표로 합쳐 보세요.
2. `customers_clean`에서 나이가 18세 미만이거나 100세 초과인 값이 있는지 확인하세요.
3. `orders_clean`에서 월별 주문 건수를 계산하세요.
4. `order_items_clean`에서 `line_total`이 큰 순서대로 상위 10개를 확인하세요.
5. 전처리 전후 행 수가 줄어든 데이터셋이 있는지 설명해 보세요.
6. LLM에게 결측치 처리 기준을 검토해 달라는 프롬프트를 직접 작성해 보세요.


In [33]:
# 과제 1. 각 데이터셋의 결측치 비율을 하나의 표로 합치기
missing_tables = []
for name, df in raw_data.items():
    table = missing_summary(df).reset_index(names='column')
    table.insert(0, 'dataset', name)
    missing_tables.append(table)

all_missing_summary = pd.concat(missing_tables, ignore_index=True)
all_missing_summary.query('missing_count > 0')

,dataset,column,missing_count,missing_ratio
0,customers,age,4,2.63
1,customers,city,2,1.32


In [34]:
# 과제 2. 18세 미만 또는 100세 초과 나이 확인
age_outliers = customers_clean.loc[
    (customers_clean['age'] < 18) | (customers_clean['age'] > 100),
    ['customer_id', 'age'],
]
age_outliers

,customer_id,age
32,33,150.0


In [35]:
# 과제 3. 월별 고유 주문 건수
monthly_order_count = (
    orders_clean.groupby('order_month', dropna=False)['order_id']
    .nunique()
    .rename('order_count')
    .reset_index()
)
monthly_order_count

,order_month,order_count
0,2025-07,19
1,2025-08,26
2,2025-09,12
3,2025-10,22
4,2025-11,23
5,2025-12,23
6,2026-01,25
7,2026-02,19
8,2026-03,22
9,2026-04,26


In [36]:
# 과제 4. line_total 상위 10개
top10_line_total = order_items_clean.nlargest(10, 'line_total')[
    ['order_item_id', 'order_id', 'product_id', 'quantity', 'unit_price', 'line_total']
]
top10_line_total

,order_item_id,order_id,product_id,quantity,unit_price,line_total
750,751,295,31,100.0,155000.0,15500000.0
737,738,290,96,50.0,96000.0,4800000.0
319,320,129,23,5.0,197000.0,985000.0
9,10,5,65,5.0,193000.0,965000.0
20,21,9,36,5.0,193000.0,965000.0
24,25,11,65,5.0,193000.0,965000.0
116,117,51,65,5.0,193000.0,965000.0
293,294,119,36,5.0,193000.0,965000.0
290,291,117,47,5.0,188000.0,940000.0
10,11,6,97,5.0,187000.0,935000.0


In [37]:
import matplotlib.pyplot as plt
from matplotlib import font_manager

IMAGE_DIR = PROJECT_ROOT / 'chapter05' / 'images'
IMAGE_DIR.mkdir(parents=True, exist_ok=True)
font_candidates = [
    Path.home() / 'research' / '프로그램 모음' / 'tools' / 'fonts' / 'noto-cjk-20230817' / 'NotoSansCJKkr-Regular.otf',
    Path('/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc'),
]
font_path = next((x for x in font_candidates if x.exists()), None)
if font_path:
    font_manager.fontManager.addfont(font_path)
    plt.rcParams['font.family'] = font_manager.FontProperties(fname=font_path).get_name()
plt.rcParams['axes.unicode_minus'] = False


def save_evidence(title, lines, filename, color='#17324d'):
    fig, ax = plt.subplots(figsize=(10.5, 6))
    ax.axis('off')
    ax.set_title(title, fontsize=20, weight='normal', color=color, pad=20)
    y = 0.88
    for text, tone in lines:
        ax.text(0.05, y, text, fontsize=13, color=tone, transform=ax.transAxes)
        y -= 0.075
    fig.tight_layout()
    fig.savefig(IMAGE_DIR / filename, dpi=160, bbox_inches='tight')
    plt.close(fig)

save_evidence('STEP 02  숫자 변환 실패 분리', [
    ('원래 age 결측                 4건', '#b45309'),
    ('숫자 변환 신규 실패           2건', '#c81e1e'),
    ('대체 전 전체 결측             6건', '#b45309'),
    ('대체 중앙값                   41', '#047857'),
    ('판정: 원래 결측과 신규 실패를 코드로 분리 확인', '#047857'),
], 'step02_cleaning.png')

save_evidence('STEP 04  관계 보류 전·후 검증', [
    ('보류 전: customer 2 / order 2 / product 2', '#c81e1e'),
    ('가격 오류 상품 4행은 삭제하지 않고 상태 표시', '#2563a6'),
    ('보류: orders 2 / order_items 18(값 오류 포함)', '#b45309'),
    ('최종 clean FK: customer 0 / order 0 / product 0', '#047857'),
    ('line_total 계산 불일치: 0', '#047857'),
], 'step04_validation.png')

shape_lines = [(f"{row.dataset:<12} {row.rows_raw:>4} -> {row.rows_processed:>4}", '#17324d') for row in comparison.itertuples()]
shape_lines.append(('판정: 감소·보류 사유를 추적 가능', '#047857'))
save_evidence('STEP 05  전처리 전후 비교', shape_lines, 'step05_before_after.png')

save_evidence('STEP 06  저장 후 재로딩·재현 검증', [
    ('clean CSV 4개 read_csv 재로딩', '#2563a6'),
    (f"재로딩 검증 {len(script_reload_validation)}개: 모두 PASS", '#047857'),
    ('shape / columns / PK / 날짜 / 계산식 / FK 확인', '#047857'),
    ('독립 스크립트 exit code: 0', '#047857'),
    ('Notebook vs Script SHA-256: 모두 동일', '#047857'),
], 'step06_reproducible.png')

print('Evidence 갱신:', [
    'step02_cleaning.png', 'step04_validation.png',
    'step05_before_after.png', 'step06_reproducible.png',
])


Evidence 갱신: ['step02_cleaning.png', 'step04_validation.png', 'step05_before_after.png', 'step06_reproducible.png']


## 제출 답안 요약

- GitHub ID: `chocoemong1`
- 보완일: 2026-11-10
- 최종 제출 URL: [chapter05.ipynb](https://github.com/chocoemong1/llm-data-analysis-study/blob/main/chapter05/chapter05.ipynb)


### 1. 전처리 전 상태

- shape: customers 152×6, products 102×4, orders 302×5, order_items 765×5
- 결측: customers의 age 4건, city 2건
- 중복: 각 데이터셋의 전체 행 및 주요 ID 중복이 각각 2건
- 타입 후보: age·price·quantity·unit_price·날짜 컬럼이 문자열

![전처리 전 상태](images/step01_before.png)

**결과 관찰:** 숫자·날짜처럼 보이는 값에 비정상 문자열이 포함되어 있고, 단순 결측 외에 변환 실패 가능성이 있었다.

**나의 해석과 판단:** 계산 전에 타입 변환 실패를 계수하고, 중복은 행 중복과 키 중복을 함께 확인해야 한다.

**업무·분석적 의미:** 기준점을 기록해야 전처리로 인한 데이터 손실을 설명할 수 있다.

**한계:** 비정상 값의 발생 원인은 원본 시스템 확인이 필요하다.

### 2. 문자열·타입·결측 처리

- age 원래 결측 **4건**과 숫자 변환 신규 실패 **2건**을 코드로 분리 출력
- 대체 전 age 결측 6건, 중앙값 41로 대체
- price 2건·quantity 5건·unit_price 2건의 숫자 변환 실패 확인
- signup_date 2건·order_date 3건의 날짜 변환 실패 확인

![타입과 결측 처리](images/step02_cleaning.png)

**결과 관찰:** `pd.to_numeric` 전후 마스크를 비교해 원래 결측과 신규 실패를 구분했다.

**나의 해석과 판단:** 신규 실패를 별도 기록해야 원래 결측 4건과 합쳐져 원인이 사라지는 문제를 막을 수 있다.


### 3. 중복·범주·이상값 처리

- 완전 중복 2건씩 제거
- 주문 상태 15개 표기를 3개로 통일
- 가격 오류 상품 4건은 상품 행을 유지하고 `price_quality_status`로 표시
- 값 계산이 불가능한 주문상세 11건은 quarantine에 보존

![중복·범주·이상값](images/step03_rules.png)

**나의 해석과 판단:** 상품을 가격 오류만으로 삭제하면 정상 `product_id`를 참조하던 주문상세 24건이 새 orphan이 된다. 따라서 상품 엔터티는 유지하고 가격만 분석에서 보류했다.


### 4. PK/FK와 파생 컬럼 검증

- 보류 전 원본 관계 오류: customer 2건, order 2건, product 2건
- 고객 없는 주문 2건과 FK 불일치 주문상세 7건을 별도 quarantine으로 분리
- 최종 clean FK: customer/order/product 모두 **0건**
- `line_total = quantity × unit_price` 불일치 0건

![관계와 파생 컬럼 검증](images/step04_validation.png)

**결과 관찰:** 가격 오류 상품을 유지해 새 product orphan 24건이 생기지 않았고, 원래 관계 오류만 보류한 뒤 clean 관계가 모두 PASS가 됐다.

**나의 해석과 판단:** 오류 행을 0으로 채우거나 삭제 사실을 숨기지 않고 별도 보류해야 clean 분석과 원인 추적을 함께 만족한다.


### 5. 전처리 전/후 비교

| 데이터 | 처리 전 | 최종 clean | 남은 결측 셀 | 변화 이유 |
| --- | ---: | ---: | ---: | --- |
| customers | 152 | 150 | 2 | 중복 제거, 잘못된 가입일 보류 |
| products | 102 | 100 | 4 | 중복 제거, 가격 오류는 행 유지·값 보류 |
| orders | 302 | 298 | 9 | 중복 및 고객 FK 오류 2건 보류 |
| order_items | 765 | 745 | 0 | 중복·숫자 오류·FK 오류 보류 |

![전처리 전후 비교](images/step05_before_after.png)

**업무·분석적 의미:** clean 테이블은 관계 일관성을 확보했으며, 제외 행은 quarantine에 남아 후속 확인이 가능하다.


### 6. 저장 결과와 재현 가능성 확인

- 저장된 clean CSV 4개를 `read_csv()`로 다시 로드
- shape·columns·주요 ID 유일/비결측·날짜 재변환·`line_total`·FK 검증 모두 PASS
- `python scripts/preprocess_data.py` 독립 재실행 성공
- Notebook 저장 결과와 스크립트 재실행 결과의 SHA-256 동일

![재현 및 재로딩 검증](images/step06_reproducible.png)

**나의 해석과 판단:** 파일 존재 확인만으로는 부족하다. 실제 재로딩 데이터의 구조와 관계를 확인해야 저장 과정의 컬럼 손실·날짜 직렬화·키 오류를 탐지할 수 있다.


### 7. 최종 판단

1. age 원래 결측과 신규 변환 실패를 별도 Evidence로 남겼다.
2. 상품 가격 오류로 정상 상품 ID를 삭제하지 않았다.
3. 관계 불일치 행을 quarantine에 보존하고 최종 clean FK를 0건으로 만들었다.
4. 저장 후 재로딩 검증과 독립 스크립트 재실행을 모두 통과했다.

#### 보완 완료 체크

- [x] age 신규 숫자 변환 실패 2건 코드·Output 검증
- [x] 새 상품 orphan 방지 및 기존 관계 오류 별도 보류
- [x] 저장 CSV 재로딩 후 내용 검증
- [x] clean FK 3종 0건
- [x] 스크립트 재실행 및 해시 동일


## 25. 정리

이번 장에서는 다음 내용을 실습했습니다.

- 원본 데이터와 전처리 데이터 분리
- 결측치 개수와 비율 확인
- 나이 결측치 중앙값 대체, 도시 결측치 Unknown 처리
- 중복 행과 주요 ID 중복 점검
- 문자열 앞뒤 공백 제거와 상태값 표기 통일
- 날짜형 변환과 주문 월/요일 파생 컬럼 생성
- 숫자형 변환과 변환 실패 건수 확인
- 이상값 후보 확인과 실습용 처리 기준 적용
- 주문 상세 금액 `line_total` 생성
- 파일 간 키 관계 재확인
- 전처리 결과 CSV와 요약 보고서 저장
- 전처리 함수를 `src/preprocessing.py` 소스 모듈로 분리

다음 장에서는 전처리된 데이터를 바탕으로 탐색적 데이터 분석, 즉 EDA를 수행합니다.
